**Part 3: Policy RAG | Step 4: Generation**  
Goal: answer each question from the retrieved clauses, with citations, or say "not found".  
Needs an API key: put `ANTHROPIC_API_KEY=...` in a `.env` file in the project root.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

# Find the project root: the folder that contains src/policy_rag.py
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "policy_rag.py").exists())
sys.path.append(str(ROOT / "src"))
import policy_rag as rag

POLICY_DIR = ROOT / "notebooks" / "RAG" / "policy"
OUT_DIR = ROOT / "notebooks" / "RAG" / "outputs"
OUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

**Step 1. Build the index and show the prompt rules.** The system prompt is the instruction the LLM follows.

In [2]:
chunks = rag.chunk_policy(rag.load_policy(POLICY_DIR / "marketplace_policy.txt"))
index = rag.PolicyIndex(chunks)
eval_df = pd.read_csv(POLICY_DIR / "eval_questions.csv")
print(rag.ANSWER_SYSTEM)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

You answer questions about a marketplace operations policy.
Rules:
1. Use ONLY the policy clauses given in the user message. Never use outside knowledge.
2. If the clauses do not contain the answer, set "answerable" to false. Do not guess.
   If a clause says the topic is handled by another document, say that.
3. Cite every clause you used by its ID, for example MOP-2.2. Cite only IDs that were provided.
4. If the question needs a calculation, show the short working using numbers from the clauses.
Reply with JSON only, no other text, in exactly this form:
{"answerable": true or false, "answer": "...", "citations": ["MOP-1.2"]}
If answerable is false, "answer" must say briefly what the policy does not cover.


**Step 2. Try one question.** Retrieve the top 3 clauses, call the LLM, show the result.

In [3]:
res = rag.answer_question(index, "Which tier is a seller with 96% on-time delivery and 11% adverse reviews?", k=3)
print(json.dumps(res, indent=2))

{
  "question": "Which tier is a seller with 96% on-time delivery and 11% adverse reviews?",
  "retrieved_ids": [
    "MOP-1.2",
    "MOP-1.3",
    "MOP-1.1"
  ],
  "top_dense": 0.7207052707672119,
  "abstained_by": null,
  "answerable": true,
  "answer": "Bronze. Gold needs OTD \u2265 95% and ARR \u2264 4%, and Silver needs OTD \u2265 88% and ARR \u2264 9%. At 11% ARR the seller fails both, since 11% > 4% and 11% > 9%. Bronze needs OTD \u2265 75% and ARR \u2264 15%, and the seller meets both (96% \u2265 75%, 11% \u2264 15%). A seller must satisfy both thresholds for a tier, so the high OTD does not lift them above Bronze. This assumes the seller has at least 15 completed orders in the trailing 90 days; otherwise they would be Unrated.",
  "citations": [
    "MOP-1.2",
    "MOP-1.3",
    "MOP-1.1"
  ],
  "invalid_citations": []
}


**Step 3. Answer all 20 questions.** Results are saved to `outputs/answers.json` for the next notebook. This makes 20 LLM calls.

In [4]:
# k=3: the 3 best clauses go to the LLM. min_dense: abstention layer 1 (tuned in notebook 05).
results = rag.run_answers(index, eval_df, k=3, mode="hybrid", min_dense=rag.MIN_DENSE_SCORE)
(OUT_DIR / "answers.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

df = pd.DataFrame(results)
df["citations"] = df["citations"].str.join(", ")
df[["qid", "gold_answerable", "answerable", "abstained_by", "citations"]]

,qid,gold_answerable,answerable,abstained_by,citations
0,Q01,1,True,NaN,"MOP-1.2, MOP-1.3"
1,Q02,1,True,NaN,MOP-1.1
2,Q03,1,True,NaN,MOP-1.5
3,Q04,1,True,NaN,"MOP-2.1, MOP-2.2, MOP-2.3"
4,Q05,1,True,NaN,"MOP-2.4, MOP-2.2"
5,Q06,1,True,NaN,"MOP-2.4, MOP-2.2, MOP-8.4"
6,Q07,1,False,llm,"MOP-3.2, MOP-3.1"
7,Q08,1,True,NaN,MOP-3.3
8,Q09,1,True,NaN,"MOP-3.1, MOP-4.1, MOP-4.3"
9,Q10,1,True,NaN,"MOP-4.2, MOP-4.1, MOP-3.1"


**Step 4. Read the answers.** Compare with `expected_answer` in the evaluation file.

In [5]:
exp = eval_df.set_index("qid")["expected_answer"]
for r in results:
    print(f"{r['qid']} | answered={r['answerable']} | cites={r['citations']}")
    print("  Q:", r["question"])
    print("  A:", r["answer"])
    print("  Expected:", exp[r["qid"]])
    print()

Q01 | answered=True | cites=['MOP-1.2', 'MOP-1.3']
  Q: A seller has 96% on-time delivery and an 11% adverse review rate. Which tier is the seller in?
  A: The seller is in the Bronze tier. A seller must satisfy both thresholds for a tier (MOP-1.3). With 96% OTD, the seller meets the Gold OTD threshold (≥95%) but not the Gold ARR threshold (≤4%), since 11% exceeds 4%. Silver requires ARR ≤9%, and 11% exceeds that too. Bronze requires OTD ≥75% and ARR ≤15%, and the seller meets both (96% ≥ 75%, 11% ≤ 15%). This matches the example in MOP-1.3.
  Expected: Bronze. Both thresholds must be met, so 11% ARR rules out Gold.

Q02 | answered=True | cites=['MOP-1.1']
  Q: How many completed orders does a seller need in the 90-day window to be given a tier?
  A: A seller needs at least 15 completed orders in the trailing 90-day window to be given a tier. Sellers with fewer than 15 completed orders in that window are classified Unrated.
  Expected: At least 15. Fewer than 15 is Unrated.

Q03 | answ